Set up

In [1]:
%pip install langchain-community pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
goo

Loading documents

In [2]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://files.eric.ed.gov/fulltext/ED536788.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

112


In [3]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0])

Introduction to  
Data Analysis 
Handbook
Migrant & Seasonal Head Start 
T echnical Assistance Center
Academy for Educational Development
“If 
I knew what 
you were going to use 
the information for  

page_content='Introduction to  
Data Analysis 
Handbook
Migrant & Seasonal Head Start 
T echnical Assistance Center
Academy for Educational Development
“If 
I knew what 
you were going to use 
the information for  
I would have done a better 
job of collecting it.”
--Famous quote from a Migrant and 
Seasonal Head Start (MSHS) staff 
person to MSHS director at a 
Community Assessment 
Training.' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 0, 'page_label': 'i'}


Splitting

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

192


Embeddings

In [5]:
%pip install langchain-huggingface sentence-transformers

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[-0.06831696629524231, 0.0007002928759902716, -0.04274025186896324, -0.042280178517103195, -0.022585870698094368, 0.05837533622980118, 0.04190812259912491, -0.036462098360061646, 0.048494499176740646, -0.001817591954022646]


Vector stores

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [9]:
ids = vector_store.add_documents(all_splits)
print(ids)

['edfaea08-3a6a-4ec8-b451-d032320aa345', 'ec93014d-2722-4c5d-b7f6-84e0fd78b162', '16398ef7-59c3-4f04-ac13-2e645e793418', '661b44fb-5111-4523-907e-8e3c9ef42ff4', '69e973a6-15a6-4129-8a6a-c865e6919784', '5f2da757-a08c-4530-8eab-e978ec41dfae', '16a2c08f-ce25-45c0-9cd4-3a004acfc9cb', '004a4fb6-477e-4211-982b-14577455f03a', 'd5a8f7df-6c2d-4247-b878-881a9a2f37ec', 'fe54112a-0f34-4976-a7aa-6f3d86f5e64a', '991a5c9f-9d70-464e-b1c1-c3c6e86fa542', 'd6c11a88-55b4-40ab-9ab7-766d665df31d', '3bc635fd-8ee8-47af-9db8-e07b20aebd95', '1fc4fe56-796a-4f82-b247-74609de955d9', '84aca67d-40b5-4e5f-9ce5-887a0a093802', '24398487-dae6-4381-8367-896c4316cb78', '938c8121-df64-4e32-923f-d6266f79b175', 'b7cc985d-3b51-41a1-a486-b54004eec7dd', 'fdf8e868-d576-4f33-bf02-26883c417b49', '3ee93664-78b5-4bc3-94b8-508f3c0c3512', '8466df73-b4d0-4d0a-a179-a24712a066a8', '040e4b22-2ef9-4356-b389-11d1703c00c7', '13b9da7a-4e5a-4751-a1de-88b688cbfe3f', '94b61e2d-33d0-4d15-a673-65ab6dfeaa19', '1fcfa3b4-7c07-4f76-824e-9c6d416a106e',

In [10]:
results = vector_store.similarity_search(
    "What does this document talk about??"
)

print(results[0])

page_content='Sample Planning Table
Purpose Questions
Data 
Collection 
Methods
Needed 
Resources Lead Person Timeframe
Purpose 1
Purpose 2
Purpose 
The planning piece of the framework gives you an opportunity to identify the resources 
that you will need. Some examples of needed resources include the translation of key 
pieces of information for parents, additional clerical support and scheduling meetings in 
conjunction with other activities.
As with other planning pieces within Head Start it is important that you consult with 
the Policy Council (PC) regarding the plan prior to its implementation. Solicit ideas and 
assistance from PC members regarding the proposed data analysis process.
Important Tip: Traditionally many programs find themselves in a crunch when dealing with 
data analysis.  Taking time to plan out your process will save you time in the long run.' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:

In [11]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("How does the Transformer encoder work?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.19506011590001104

page_content='© AED/TAC-12 Spring 2006. 0' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 95, 'page_label': '90', 'start_index': 0}


In [12]:
embedding = embeddings.embed_query("What are the advantages of the Transformer over RNNs?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='© AED/TAC-12 Spring 2006. 6
Appendix C: Supplemental                                          Resources' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 101, 'page_label': '96', 'start_index': 0}


Retrievers

In [13]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "What is the main architecture proposed in the pdf?",
        "What is data analysis?",

    ],
)

[[Document(id='5f252742-4e69-4998-b8f5-d3372b639cd4', metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 29, 'page_label': '24', 'start_index': 819}, page_content='Sample Planning Table\nPurpose Questions\nData \nCollection \nMethods\nNeeded \nResources Lead Person Timeframe\nPurpose 1\nPurpose 2\nPurpose \x18\nThe planning piece of the framework gives you an opportunity to identify the resources \nthat you will need. Some examples of needed resources include the translation of key \npieces of information for parents, additional clerical support and scheduling meetings in \nconjunction with other activities.\nAs with other planning pieces within Head Start it is important that you consult with \nthe Policy Council (PC) regarding the plan prior to its impleme